# Lab 2 — Part B: Evaluation and data preparation strategies

This notebook evaluates the baseline XGBoost classifier (metrics, confusion matrix, ROC and precision-recall curves, classification thresholds, trees, feature importance) and compares different data preparation strategies and methods for imbalanced classes.

All experiments use the shared data preparation from `lab2_utils.py`, the fixed stratified 80/20 split and stratified cross-validation on the training set. The test set is only used to evaluate the baseline model; all decisions about the data preparation are made with cross-validation.

## 1. Load data and stratified train-test split 80% - 20%

First, I load the prepared Ames Housing dataset, which contains 2,930 houses and 333 features. The target variable is `PriceClass`, which represents the six price categories we want to predict.

I split the data into training (80%) and test (20%) sets. I use stratification (`stratify=y`) to make sure that all six price classes are represented in approximately the same proportions in both sets. This is especially important because the classes are imbalanced and some of them have considerably fewer observations.

I use `random_state=42` to make the split reproducible.

The training set will be used for training and cross-validation, while the test set will remain separate and will only be used for evaluation.

**Note:** in this notebook, `X_train`, `y_train`, `X_test` and `y_test` always refer to the baseline data. Every variant of the data gets its own variable name, so that all experiments are compared with the same reference.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost, optuna, plotly
from xgboost import XGBClassifier
from lab2_utils import (load_data, clean_data, encode_features, make_target,
                        get_data, get_split, CLASS_NAMES)

print(xgboost.__version__, optuna.__version__, plotly.__version__)

X, y = get_data()
X_train, X_test, y_train, y_test = get_split(X, y)
print("X:", X.shape)
print(y_train.value_counts().sort_index())
print(y_test.value_counts().sort_index())

## 2. Stratified 5-Fold Cross-Validation

To get a better idea of how the model performs on different subsets of the data, I use stratified 5-fold cross-validation on the training set.

The training data is divided into five folds. In each iteration, four folds are used for training and the remaining one for validation. This process is repeated five times, so each fold is used for validation once.

I use **macro-averaged F1-score (macro-F1)** as the evaluation metric because we have six imbalanced price classes. For each fold, the F1-score is calculated separately for all six classes and then averaged, giving every class equal importance regardless of how many houses it contains.

After completing all five iterations, I look at:

- **Individual fold scores:** Macro-F1 obtained in each of the five iterations.
- **Mean macro-F1:** Average of the five fold scores, giving an idea of the model's overall performance.
- **Standard deviation:** Shows how much the scores vary between folds. A lower value means that the model performs more consistently across the different subsets.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Define 5-fold stratified cross-validation
cv5 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Evaluate the baseline model using 5-fold CV
cv_scores = cross_val_score(
    XGBClassifier(random_state=42),
    X_train,
    y_train,
    cv=cv5,
    scoring="f1_macro"
)

print("Individual fold scores:", cv_scores.round(3))
print("Mean macro-F1:", round(cv_scores.mean(), 3))
print("Standard deviation:", round(cv_scores.std(), 3))

**Interpretation:** The baseline reaches a mean macro-F1 of **0.722 ± 0.020**. The five fold scores range from 0.700 to 0.752, so a single split can differ from the average by about 0.03. This value is the reference for all experiments in this notebook.

## 3. Training the Baseline Model and Generating Predictions

After cross-validation, I train a new XGBoost model using the entire training set (80% of the data). The previous five models were only used to evaluate the performance through cross-validation. Now I want to have one trained model that I can use for the actual evaluation.

After training, I make predictions on the test set (20% of the data), which the model has not seen before.

I use two types of predictions:
- `y_pred`: the predicted price class for each house.
- `y_prob`: the predicted probabilities for each of the six price classes.

I need the predicted classes for the evaluation metrics and confusion matrix, while the probabilities are used for the ROC and Precision-Recall curves and for finding the best classification thresholds.

In [ ]:
baseline = XGBClassifier(random_state=42)

baseline.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import f1_score

y_pred = baseline.predict(X_test)

test_macro_f1 = f1_score(y_test, y_pred, average="macro")

print("Test macro-F1:", round(test_macro_f1, 3))

## 4. Evaluation Metrics

Now I want to evaluate how well the model predicts the price classes on the test set. I use the predictions from the baseline model and compare them with the actual price classes.

I calculate the following metrics:

- **Accuracy:** The proportion of houses for which the predicted price class is correct.
- **Precision:** How many of the houses predicted to belong to a certain class actually belong to that class.
- **Recall:** How many of the actual houses in a class were correctly identified by the model.
- **F1-score:** Combines precision and recall into one metric (their harmonic mean).

Since the dataset is imbalanced, I use macro-averaging for precision, recall and F1. This means that all six price classes have equal importance when calculating the average.

I also look at the individual class scores to see which price categories are more difficult for the model to predict.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Generate predictions
y_pred = baseline.predict(X_test)
y_prob = baseline.predict_proba(X_test)

# Calculate evaluation metrics
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Macro Precision:", round(precision_score(y_test, y_pred, average="macro"), 3))
print("Macro Recall:", round(recall_score(y_test, y_pred, average="macro"), 3))
print("Macro F1:", round(f1_score(y_test, y_pred, average="macro"), 3))

# Individual metrics for each price class
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=range(len(CLASS_NAMES)),
    target_names=CLASS_NAMES,
    digits=3,
    zero_division=0
))

**Interpretation:**
- **Accuracy (0.782) is higher than macro-F1 (0.748)** because accuracy is dominated by the large classes, while macro-F1 counts every class equally.
- **Macro precision (0.778) is higher than macro recall (0.727):** when the model predicts a small class, it is usually right, but it misses many of its houses. The clearest case is ≥400k: precision 0.778, but recall only 0.538 (7 of 13 houses found).
- The best classes are 250–400k (F1 = 0.840) and 130–180k (0.818); the weakest are ≥400k (0.636) and 100–130k (0.676).

## 5. Confusion Matrix

Next, I want to see which price classes the model confuses with each other. For this, I create a confusion matrix comparing the actual price classes with the predicted ones.

The confusion matrix has six rows and six columns because we have six possible classes.

- **Rows:** Actual price classes.
- **Columns:** Predicted price classes.
- **Diagonal:** Houses that were classified correctly.
- **Off-diagonal:** Houses that were assigned to the wrong price class.

This should help me see whether the model mainly confuses neighboring price categories or also makes larger classification errors.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=range(len(CLASS_NAMES))
)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Oranges",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES
)

plt.xlabel("Predicted price class")
plt.ylabel("Actual price class")
plt.title("Confusion Matrix - Baseline XGBoost")
plt.tight_layout()
plt.show()

# How many errors go only to the neighboring class?
errors = cm.sum() - np.trace(cm)
neighbor_errors = sum(cm[i, i + 1] + cm[i + 1, i] for i in range(len(CLASS_NAMES) - 1))
print(f"Misclassified test houses: {errors}, of which {neighbor_errors} went to a neighboring class")

**Interpretation:**
- 122 of the 128 misclassified test houses were put into a **neighboring** price class. The classes are cuts of a continuous price, so houses close to a class boundary are hard to separate.
- **≥400k:** 6 of the 13 houses were predicted as 250–400k, which explains the low recall of this class.
- **100–130k** is confused in both directions (9 houses as <100k, 24 as 130–180k). With a width of only $30,000, it has close neighbors on both sides.
- Only 6 errors skip a class; one house of 130–180k was even predicted as ≥400k.

## 6. ROC Curve

I also want to investigate how well the model can distinguish between the different price classes using ROC curves.

Since this is a multiclass classification problem, I use the One-vs-Rest (OvR) approach. This means that I evaluate each price class separately, treating it as the positive class and all the other classes as negative.

For each class, I calculate:
- **True Positive Rate (TPR):** The proportion of actual positive observations correctly identified.
- **False Positive Rate (FPR):** The proportion of negative observations incorrectly classified as positive.
- **AUC:** The area under the ROC curve, which gives an overall measure of how well the model separates the class from the others.

An AUC closer to 1 indicates better class separation, while a value around 0.5 indicates performance similar to random ranking.

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Convert the actual classes into binary format (One-vs-Rest)
y_test_bin = label_binarize(
    y_test,
    classes=np.arange(len(CLASS_NAMES))
)

plt.figure(figsize=(10, 7))

for i, class_name in enumerate(CLASS_NAMES):
    fpr, tpr, thresholds = roc_curve(
        y_test_bin[:, i],
        y_prob[:, i]
    )

    roc_auc = auc(fpr, tpr)

    plt.plot(
        fpr,
        tpr,
        label=f"{class_name} (AUC = {roc_auc:.3f})"
    )

plt.plot([0, 1], [0, 1], "k--", label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves - One-vs-Rest")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

**Interpretation:**
- All AUC values are high (0.919–0.995): for every class, the model ranks most of its houses above the houses of the other classes.
- **≥400k has the highest AUC (0.995) but the lowest F1 (0.636).** The expensive houses do receive the highest ≥400k probabilities, but for many of them another class still has a higher probability, so the argmax decision misses them. The problem is therefore the **decision rule**, not the ranking. This motivates class-specific thresholds (Section 8) and class weights (Section 13).
- The middle classes (130–180k: 0.919, 100–130k: 0.930) have the lowest AUC because they have neighboring classes on both sides.
- With imbalanced classes, ROC curves can look optimistic: for ≥400k there are 573 negatives and only 13 positives, so even several false positives give a low false positive rate. The precision-recall curves below show this more clearly.

## 7. Precision-Recall Curve

Besides ROC curves, I use Precision-Recall curves to investigate the model's performance for each price class.

The curves show the relationship between precision and recall at different classification thresholds. I use the same One-vs-Rest approach as with the ROC curves, treating each price class separately.

This is particularly interesting for our dataset because the classes are imbalanced. Precision-Recall curves can give us a better idea of how well the model identifies the smaller classes, especially the ≥400k category.

I also calculate the Average Precision (AP) for each class to summarize the corresponding curve.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

plt.figure(figsize=(10, 7))

for i, class_name in enumerate(CLASS_NAMES):
    precision, recall, thresholds = precision_recall_curve(
        y_test_bin[:, i],
        y_prob[:, i]
    )

    ap = average_precision_score(
        y_test_bin[:, i],
        y_prob[:, i]
    )

    plt.plot(
        recall,
        precision,
        label=f"{class_name} (AP = {ap:.3f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves - One-vs-Rest")
plt.legend(loc="best")
plt.grid(alpha=0.3)
plt.show()

**Interpretation:**
- For the small classes, the average precision (AP) is clearly lower than the AUC: ≥400k drops from AUC 0.995 to **AP 0.776**. With only 13 positive houses, a few false positives already lower the precision strongly, which the ROC curve hides.
- 100–130k has the lowest AP (0.701), consistent with its low F1 and with the confusion matrix.
- The large and well-separated classes keep high AP values (250–400k: 0.914, 180–250k: 0.878).

## 8. Finding the Best Classification Thresholds

Until now, the model assigns each house to the class with the highest predicted probability. However, this does not necessarily give us the best macro-F1, especially because the price classes are imbalanced.

I want to investigate whether using different thresholds for each class can improve the predictions.

To do this, I first generate out-of-fold probabilities using 5-fold cross-validation. This means that every training observation receives predictions from a model that did not use that observation for training.

I then search for class-specific thresholds that improve the macro-F1 score. The threshold of a class is subtracted from its probability, and the class with the highest adjusted value is predicted. The search changes the threshold of one class at a time and keeps a change if macro-F1 improves.

- **Default prediction:** The class with the highest predicted probability.
- **Threshold-adjusted prediction:** The class with the highest probability after accounting for its individual threshold.

I compare the resulting macro-F1 scores to see whether adjusting the thresholds improves the classification.

In [ ]:
from sklearn.model_selection import cross_val_predict

oof_prob = cross_val_predict(
    XGBClassifier(random_state=42),
    X_train,
    y_train,
    cv=cv5,
    method="predict_proba"
)

print("OOF probabilities shape:", oof_prob.shape)

In [ ]:
def predict_with_thresholds(probabilities, thresholds):
    """Class with the highest probability after subtracting its class-specific threshold."""
    adjusted_scores = probabilities - thresholds
    return np.argmax(adjusted_scores, axis=1)


def search_thresholds(probabilities, y_true, candidates=np.linspace(0.05, 0.95, 19), rounds=3):
    """Greedy search: change the threshold of one class at a time and keep it if macro-F1 improves."""
    best_thresholds = np.full(len(CLASS_NAMES), 0.5)  # equal thresholds = normal argmax prediction
    best_score = f1_score(y_true, predict_with_thresholds(probabilities, best_thresholds), average="macro")

    for _ in range(rounds):
        for class_idx in range(len(CLASS_NAMES)):
            for threshold in candidates:
                trial_thresholds = best_thresholds.copy()
                trial_thresholds[class_idx] = threshold
                score = f1_score(
                    y_true,
                    predict_with_thresholds(probabilities, trial_thresholds),
                    average="macro"
                )
                if score > best_score:
                    best_score = score
                    best_thresholds = trial_thresholds

    return best_thresholds, best_score


best_thresholds, best_score = search_thresholds(oof_prob, y_train)

print("Best thresholds:")
for name, threshold in zip(CLASS_NAMES, best_thresholds):
    print(f"{name}: {threshold:.3f}")

print("\nDefault OOF macro-F1:",
      round(f1_score(y_train, np.argmax(oof_prob, axis=1), average="macro"), 3))

print("Threshold-adjusted OOF macro-F1:",
      round(best_score, 3))

As a check only, I also apply the thresholds to the test set. No decision is based on this result.

In [ ]:
# Check only: apply the thresholds to the test probabilities
y_pred_threshold = predict_with_thresholds(
    y_prob, best_thresholds
)

print("Default test macro-F1:",
      round(f1_score(y_test, y_pred, average="macro"), 3))

print("Threshold-adjusted test macro-F1:",
      round(f1_score(y_test, y_pred_threshold, average="macro"), 3))

**Interpretation:**
- On the out-of-fold predictions, the thresholds raise macro-F1 from 0.723 to **0.744**, but on the test set it drops from 0.748 to **0.736**.
- The thresholds were searched and evaluated on the **same** predictions: six thresholds are fitted to 2,344 houses, of which only 51 belong to ≥400k, so the search also adapts to random fluctuations (overfitting). Two thresholds even ended at the edge of the search grid (0.95).
- Whether thresholds help on unseen data is checked properly in Section 15.

## 9. Visualizing the First and Best Tree

I also want to take a look at the decision trees created by XGBoost to get a better understanding of how the model makes its predictions.

Since we have six price classes, each boosting iteration generates six individual trees, one for each class.

I first visualize the initial tree from the baseline model. To identify the best iteration, I train a separate model with early stopping using an additional validation subset taken from the training data.

Early stopping monitors the validation loss and stops training when it no longer improves for a specified number of iterations.

For the visualization, I select the tree belonging to the first class in the best boosting iteration, i.e. the last tree that still improved the validation loss. This allows me to compare an initial tree with one from a later, best-performing iteration.

### First tree

In [ ]:
from xgboost import plot_tree

fig, ax = plt.subplots(figsize=(35, 20))

plot_tree(
    baseline,
    tree_idx=0,
    rankdir="LR",
    ax=ax
)

ax.set_title("First XGBoost Tree (Class 0)")
plt.tight_layout()
plt.show()

### Train a model with early stopping

In [ ]:
from sklearn.model_selection import train_test_split

# Create an internal validation split from the training data
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

# Train a separate model with early stopping
tree_model = XGBClassifier(
    random_state=42,
    n_estimators=1000,
    eval_metric="mlogloss",
    early_stopping_rounds=30
)

tree_model.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print("Best iteration:", tree_model.best_iteration)
print("Best validation loss:", tree_model.best_score)

### Visualize the best tree

In [ ]:
# Find the index of the first tree (class 0) in the best boosting iteration
best_tree_idx = tree_model.best_iteration * len(CLASS_NAMES)

fig, ax = plt.subplots(figsize=(45, 25))

plot_tree(
    tree_model,
    tree_idx=best_tree_idx,
    rankdir="LR",
    ax=ax
)

ax.set_title(
    f"Best XGBoost Tree (Iteration {tree_model.best_iteration}, Class 0)"
)

plt.tight_layout()

# Save a high-resolution version so you can zoom in
fig.savefig("best_xgboost_tree.png", dpi=300, bbox_inches="tight")

plt.show()

### How large are the corrections of later trees?

To compare the first and the later trees in numbers, I look at the mean absolute leaf value of the class-0 tree in every boosting iteration of the early-stopping model.

In [ ]:
# How large are the corrections of the trees over the boosting iterations? (class 0)
trees = tree_model.get_booster().trees_to_dataframe()
leaves = trees[trees["Feature"] == "Leaf"]
class0_leaves = leaves[leaves["Tree"] % len(CLASS_NAMES) == 0]
mean_abs_leaf = class0_leaves.groupby("Tree")["Gain"].apply(lambda v: v.abs().mean())
iterations = mean_abs_leaf.index // len(CLASS_NAMES)

plt.figure(figsize=(9, 5))
plt.plot(iterations, mean_abs_leaf.values, "o-")
plt.axvline(tree_model.best_iteration, color="red", linestyle="--", label="Best iteration")
plt.xlabel("Boosting iteration")
plt.ylabel("Mean absolute leaf value (class 0)")
plt.title("Size of the corrections made by each tree")
plt.legend()
plt.show()

print("Mean |leaf value|, first tree:", round(mean_abs_leaf.iloc[0], 3))
print("Mean |leaf value|, best tree:", round(mean_abs_leaf.loc[best_tree_idx], 3))

**Interpretation:**
- The **first tree** (class <100k) starts with the most basic distinctions: `Overall Qual`, basement and living area, garage size.
- The **best tree** (iteration 22) splits on much more specific features (e.g. fireplaces, single neighborhoods, fence quality), and its leaf values are much smaller. In the early-stopping model, the mean absolute leaf value drops from 0.391 in the first tree to 0.068 in the best tree: each later tree only corrects the remaining errors of the previous ones, which is the idea of boosting.
- **Early stopping** finds the best validation loss after only 22 iterations (with the default learning rate 0.3), while the baseline uses 100 trees. The additional trees mainly fit the training data, which matches the strong overfitting of the baseline (training macro-F1 = 1.000, Part A). `n_estimators`, `learning_rate` and early stopping are therefore important for the tuning (Part C).

## 10. Feature Importance

Next, I want to see which features have the greatest influence on the model. For this, I compare two different feature importance measurements provided by XGBoost.

- **Gain:** Measures how much a feature contributes to improving the predictions when it is used in a split (average over all its splits). A higher gain means that the feature provides more information for the model.
- **Split:** Counts how often a feature is used in the decision trees. A feature can be used frequently without necessarily contributing a lot to the predictions.

I compare the top features according to both measurements to see whether the most frequently used features are also the ones that contribute the most to the model.

In [ ]:
booster = baseline.get_booster()

# Feature importance based on gain (average gain per split)
gain_importance = pd.Series(
    booster.get_score(importance_type="gain")
).sort_values(ascending=False)

top_gain = gain_importance.head(15).sort_values()

plt.figure(figsize=(10, 7))
top_gain.plot(kind="barh")
plt.xlabel("Average Gain")
plt.ylabel("Feature")
plt.title("Top 15 Features by Gain")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance based on the number of splits
split_importance = pd.Series(
    booster.get_score(importance_type="weight")
).sort_values(ascending=False)

top_split = split_importance.head(15).sort_values()

plt.figure(figsize=(10, 7))
top_split.plot(kind="barh")
plt.xlabel("Number of Splits")
plt.ylabel("Feature")
plt.title("Top 15 Features by Split")
plt.tight_layout()
plt.show()

common = set(gain_importance.head(15).index) & set(split_importance.head(15).index)
print("Features in both top-15 lists:", sorted(common))

**Interpretation:**
- **Gain** (average improvement per split) is highest for one-hot dummies and quality features such as `Bsmt Qual_Gd`, `Overall Qual`, `Neighborhood_BrDale` and `MS SubClass_160`. These features are used rarely, but when they are used, they separate the classes strongly.
- **Split count** is dominated by continuous area features (`Gr Liv Area`, `Lot Area`, `Garage Area`, `Lot Frontage`, `Total Bsmt SF`). They offer many possible split points and are used for many small adjustments, so they are counted often although each split gains little.
- `Overall Qual` is the only feature in both top-15 lists: it is used often **and** its splits are informative.
- With one-hot encoding, the importance of a categorical feature is divided between its dummies (e.g. 28 neighborhoods), so features like `Neighborhood` look less important than they are. This is one argument for ordinal encoding of ordered categories (Section 12.3).

## 11. Feature Engineering Experiments

I want to investigate whether the engineered features from Lab 1 improve the performance of the baseline XGBoost model.

The features are computed here with the same definitions as in Lab 1 (Part 3.3), including the corrections found there: negative ages are set to 0 (houses sold before they were finished), and `Year Remod/Add` = 1950 for houses built before 1950 is not counted as a remodel.

I will compare the original dataset with different combinations of engineered features, using:
- The same stratified 80/20 train-test split.
- Stratified 5-fold cross-validation on the training set.
- The same XGBoost model parameters.
- Macro-F1 as the primary evaluation metric.

This allows me to compare the results while keeping the other experimental conditions unchanged.

In [ ]:
ENGINEERED_FEATURES = ["HouseAge", "YearsSinceRemodel", "WasRemodeled", "TotalArea", "TotalBathrooms"]


def add_engineered_features(df, features):
    """Engineered features from Lab 1 (Part 3.3), including the corrections found there."""
    df = df.copy()
    if "HouseAge" in features:  # houses sold before they were finished -> age 0
        df["HouseAge"] = (df["Yr Sold"] - df["Year Built"]).clip(lower=0)
    if "YearsSinceRemodel" in features:
        df["YearsSinceRemodel"] = (df["Yr Sold"] - df["Year Remod/Add"]).clip(lower=0)
    if "WasRemodeled" in features:  # Year Remod/Add = 1950 for older houses means "no information"
        artefact = (df["Year Built"] < 1950) & (df["Year Remod/Add"] == 1950)
        df["WasRemodeled"] = ((df["Year Remod/Add"] > df["Year Built"]) & ~artefact).astype(int)
    if "TotalArea" in features:
        df["TotalArea"] = df["Gr Liv Area"] + df["Total Bsmt SF"]
    if "TotalBathrooms" in features:
        df["TotalBathrooms"] = (df["Full Bath"] + df["Bsmt Full Bath"]
                                + 0.5 * (df["Half Bath"] + df["Bsmt Half Bath"]))
    return df


df_clean = clean_data(load_data())
y_all = make_target(df_clean)


def cv_macro_f1(X_variant):
    """Stratified 5-fold CV on the training part of the fixed 80/20 split."""
    X_variant_train, _, y_variant_train, _ = get_split(X_variant, y_all)
    return cross_val_score(XGBClassifier(random_state=42), X_variant_train, y_variant_train,
                           cv=cv5, scoring="f1_macro")


def run_feature_experiments(experiments):
    rows = []
    for name, selected_features in experiments.items():
        X_variant = encode_features(add_engineered_features(df_clean, selected_features))
        scores = cv_macro_f1(X_variant)
        rows.append({"Experiment": name,
                     "Mean CV Macro-F1": scores.mean(),
                     "Std CV Macro-F1": scores.std()})
        print(f"{name}: {scores.mean():.3f} +/- {scores.std():.3f}")
    table = pd.DataFrame(rows)
    table["Difference vs Baseline"] = (
        table["Mean CV Macro-F1"] - table.loc[table["Experiment"] == "Baseline", "Mean CV Macro-F1"].iloc[0]
    )
    return table.sort_values("Mean CV Macro-F1", ascending=False).round(4)

In [ ]:
experiments = {
    "Baseline": [],
    "HouseAge": ["HouseAge"],
    "YearsSinceRemodel": ["YearsSinceRemodel"],
    "WasRemodeled": ["WasRemodeled"],
    "TotalArea": ["TotalArea"],
    "TotalBathrooms": ["TotalBathrooms"],
    "TotalArea + TotalBathrooms": ["TotalArea", "TotalBathrooms"],
    "All five features": ENGINEERED_FEATURES,
}

fe_results_1 = run_feature_experiments(experiments)
display(fe_results_1)

Next, I combine the best-performing individual features.

In [ ]:
experiments = {
    "Baseline": [],

    # Best individual features
    "TotalArea": ["TotalArea"],
    "HouseAge": ["HouseAge"],
    "TotalBathrooms": ["TotalBathrooms"],

    # Combinations of the best-performing features
    "TotalArea + HouseAge": ["TotalArea", "HouseAge"],
    "TotalArea + TotalBathrooms": ["TotalArea", "TotalBathrooms"],
    "HouseAge + TotalBathrooms": ["HouseAge", "TotalBathrooms"],

    # All three best features together
    "Top three features": ["TotalArea", "HouseAge", "TotalBathrooms"],
}

fe_results_2 = run_feature_experiments(experiments)
display(fe_results_2)

**Interpretation:**
- `TotalArea` (+0.015), `HouseAge` (+0.014) and `TotalBathrooms` (+0.009) improve the score on their own; `YearsSinceRemodel` (+0.003) and `WasRemodeled` (−0.003) do not.
- The three best features together reach **0.737 ± 0.029**, practically the same as `TotalArea` alone (0.737). Combinations do not simply add up: `HouseAge + TotalBathrooms` (0.717) is even worse than each of them alone.
- These differences are close to the noise level of a single 5-fold CV (~0.01, Part A), and the standard deviations are large (up to 0.035). The combination is therefore checked again with repeated CV in Section 14.

## 12. Data Preparation Strategies

I compare three changes to the data preparation with the baseline: removing outliers, imputing missing values and ordinal encoding.

All strategies are evaluated on the same training set and the same 5 folds as the baseline. Steps that change the training data (removing outliers, imputing, and later balancing the classes) are applied **only to the training part of each fold**, so the validation folds stay unchanged. Otherwise, information from the validation data could leak into the training, or the strategies would be evaluated on different data.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.utils import resample


def evaluate_strategy(X_tr, y_tr, cv=cv5, remove_outliers=False, impute=False,
                      weights=False, oversample=False):
    """Macro-F1 per fold. Every change is applied only to the training part of each fold."""
    X_tr, y_tr = X_tr.reset_index(drop=True), y_tr.reset_index(drop=True)
    scores = []
    for fold, (fold_train, fold_val) in enumerate(cv.split(X_tr, y_tr)):
        if remove_outliers:  # houses > 4000 sqft, removed only from the training part
            fold_train = fold_train[X_tr.iloc[fold_train]["Gr Liv Area"].values <= 4000]

        X_fold_train, y_fold_train = X_tr.iloc[fold_train], y_tr.iloc[fold_train]
        X_fold_val, y_fold_val = X_tr.iloc[fold_val], y_tr.iloc[fold_val]

        if oversample:  # duplicate houses of the smaller classes up to the size of the largest class
            max_count = y_fold_train.value_counts().max()
            positions = np.concatenate([
                resample(np.where(y_fold_train.to_numpy() == label)[0], replace=True,
                         n_samples=max_count, random_state=42 + fold)
                for label in sorted(y_fold_train.unique())
            ])
            X_fold_train, y_fold_train = X_fold_train.iloc[positions], y_fold_train.iloc[positions]

        if impute:  # medians are computed on the training part only
            imputer = SimpleImputer(strategy="median").fit(X_fold_train)
            X_fold_train, X_fold_val = imputer.transform(X_fold_train), imputer.transform(X_fold_val)

        sample_weight = compute_sample_weight("balanced", y_fold_train) if weights else None
        model = XGBClassifier(random_state=42)
        model.fit(X_fold_train, y_fold_train, sample_weight=sample_weight)
        scores.append(f1_score(y_fold_val, model.predict(X_fold_val), average="macro"))
    return np.array(scores)


baseline_scores = evaluate_strategy(X_train, y_train)
print(f"Baseline (same folds as before): {baseline_scores.mean():.4f} +/- {baseline_scores.std():.4f}")

### 12.1 Outlier Removal

In Lab 1, we identified five houses with more than 4,000 sqft of above-ground living area. Three were unfinished houses sold before construction was completed, while the other two were unusually large and expensive houses.

I want to investigate whether removing these observations improves the classification model.

The houses are removed **only from the training data**. The test set and the validation folds must stay unchanged: a model in practice also has to predict such houses, and only then are the results comparable with the rest of the lab.

I also need to keep in mind that two of these houses are expensive properties, so removing them also affects the already-small ≥400k price class.

In [ ]:
print("Houses > 4000 sqft in the whole dataset:", (df_clean["Gr Liv Area"] > 4000).sum())
print("Houses > 4000 sqft in the training set:", (X_train["Gr Liv Area"] > 4000).sum())
print("Houses > 4000 sqft in the test set:    ", (X_test["Gr Liv Area"] > 4000).sum())

outlier_scores = evaluate_strategy(X_train, y_train, remove_outliers=True)

print(f"\nBaseline:                 {baseline_scores.mean():.4f} +/- {baseline_scores.std():.4f}")
print(f"Without outliers (train): {outlier_scores.mean():.4f} +/- {outlier_scores.std():.4f}")

**Interpretation:**
- Of the five houses above 4,000 sqft, two are in the training set and three in the test set. Removing them from the whole dataset before the split would have removed three test houses, so the test set would no longer be the one used in the other parts.
- Removed only from the training folds, the score rises from 0.7225 to 0.7341. However, only two houses are removed, and the repeated CV in Section 14 shows that this difference is noise (+0.000 over 15 fits).

### 12.2 Handling Missing Values

In the original data preparation, the remaining numerical missing values are left as NaN because XGBoost can handle missing values natively: at every split, it learns in which direction houses with a missing value should go.

I want to compare this approach with median imputation, where missing numerical values are replaced by the median of the corresponding feature. The medians are calculated separately within each training fold, so that no information from the validation fold is used during preprocessing.

In [ ]:
imputed_scores = evaluate_strategy(X_train, y_train, impute=True)

print(f"Native missing values: {baseline_scores.mean():.4f} +/- {baseline_scores.std():.4f}")
print(f"Median imputation:     {imputed_scores.mean():.4f} +/- {imputed_scores.std():.4f}")

**Interpretation:** Median imputation does not help (0.7201 vs. 0.7225, a difference within the noise). XGBoost's native handling of missing values works at least as well: it can learn that a missing value carries information (e.g. a missing `Garage Yr Blt` means that there is no garage), which a median would hide. The missing values are therefore left as NaN.

### 12.3 Ordinal vs. One-Hot Encoding

In the baseline model, all categorical features are encoded using one-hot encoding.

However, some categorical features describe quality levels that already have a natural order, such as Poor, Fair, Typical, Good and Excellent.

For these features, I want to investigate whether ordinal encoding, where each category is represented by a number according to its order, performs better than one-hot encoding.

I keep the remaining nominal features one-hot encoded, since they do not have a meaningful order.

In [ ]:
# Define the natural order of the categorical features
quality = {"NA": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

ordinal_mappings = {
    # Quality-related features
    "Exter Qual": quality,
    "Exter Cond": quality,
    "Bsmt Qual": quality,
    "Bsmt Cond": quality,
    "Heating QC": quality,
    "Kitchen Qual": quality,
    "Fireplace Qu": quality,
    "Garage Qual": quality,
    "Garage Cond": quality,
    "Pool QC": quality,

    # Basement features
    "Bsmt Exposure": {"NA": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFin Type 1": {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFin Type 2": {"NA": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},

    # Other ordered features
    "Lot Shape": {"IR3": 0, "IR2": 1, "IR1": 2, "Reg": 3},
    "Utilities": {"ELO": 0, "NoSeWa": 1, "NoSewr": 2, "AllPub": 3},
    "Land Slope": {"Sev": 0, "Mod": 1, "Gtl": 2},
    "Functional": {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
    "Garage Finish": {"NA": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Paved Drive": {"N": 0, "P": 1, "Y": 2},
    "Fence": {"NA": 0, "MnWw": 1, "GdWo": 2, "MnPrv": 3, "GdPrv": 4},
}


def ordinal_encode(df):
    """Replace ordered categories by numbers; check for unexpected categories first."""
    df = df.copy()
    for col, mapping in ordinal_mappings.items():
        unknown = set(df[col].dropna().unique()) - set(mapping)
        if unknown:
            raise ValueError(f"Unexpected categories in {col}: {unknown}")
        df[col] = df[col].map(mapping)
    return df


# Remaining nominal features are one-hot encoded
X_ordinal = encode_features(ordinal_encode(df_clean))
print("Number of features: one-hot", X.shape[1], "| ordinal", X_ordinal.shape[1])

ordinal_scores = cv_macro_f1(X_ordinal)
onehot_scores = cv_macro_f1(X)

print(f"Ordinal encoding: {ordinal_scores.mean():.4f} +/- {ordinal_scores.std():.4f}")
print(f"One-hot encoding: {onehot_scores.mean():.4f} +/- {onehot_scores.std():.4f}")
print(f"Difference: {ordinal_scores.mean() - onehot_scores.mean():+.4f}")

**Interpretation:**
- Ordinal encoding reduces the number of features from 333 to 250 and gives a slightly higher and more stable score (0.7281 ± 0.016 vs. 0.7225 ± 0.020).
- The difference (+0.006) is within the noise of a single CV, but ordinal encoding is also the more appropriate encoding for ordered categories (Ex > Gd > TA > Fa > Po): a single split such as `Kitchen Qual < 4` separates all good and excellent kitchens from the others, while one-hot encoding needs several splits for this.

## 13. Handling Imbalanced Classes

As seen in the beginning, the six price classes are not equally represented in the dataset. The ≥400k class has only 64 houses, while the 130–180k class has 1,037.

Although macro-F1 gives every class equal importance during evaluation, the model is not automatically trained with equal importance for every class.

I want to investigate whether balancing the training data can improve the predictions, especially for the smaller classes.

I compare two approaches:

- **Balanced sample weights:** Each training observation receives a weight based on the frequency of its class. Observations from smaller classes receive higher weights, so errors on these examples have a greater influence during training.
- **Random oversampling:** Houses from the smaller classes are randomly selected with replacement until all classes have the same number of observations.

I compare both approaches with the original baseline using the same stratified 5 folds. Balancing is performed only on the training folds, so the validation data remains unchanged.

In [ ]:
weight_scores = evaluate_strategy(X_train, y_train, weights=True)
oversampling_scores = evaluate_strategy(X_train, y_train, oversample=True)

imbalance_results = pd.DataFrame([
    {"Strategy": "Original baseline", "Mean macro-F1": baseline_scores.mean(), "Std": baseline_scores.std()},
    {"Strategy": "Balanced sample weights", "Mean macro-F1": weight_scores.mean(), "Std": weight_scores.std()},
    {"Strategy": "Random oversampling", "Mean macro-F1": oversampling_scores.mean(), "Std": oversampling_scores.std()},
])

display(imbalance_results.round(4))

**Interpretation:**
- **Balanced sample weights** raise the score from 0.7225 to **0.7356** and reduce the spread (0.015). Errors on the small classes count more during training, which directly targets macro-F1.
- **Random oversampling** does not change the mean (0.7224). Duplicating the same 51 expensive houses many times mostly makes the model memorize them; it adds no new information.
- On its own, the effect of the weights is again close to the noise level; its real contribution is measured in Section 14.

## 14. Combining the Strategies

Sections 11–13 tested each strategy on its own, with a single 5-fold cross-validation. Two problems remain:

1. **Noise:** in Part A, repeating the same 5-fold CV with a different shuffle moved the score by up to ~0.01. Several of the differences above are of this size.
2. **Combination:** effects do not simply add up. For example, `HouseAge` and `TotalBathrooms` each improve the score on their own, but not together (Section 11).

Therefore, I evaluate the strategies again with **repeated stratified 5-fold CV (5 folds × 3 repeats = 15 fits)**: each strategy alone, all five together, and all five without one of them (*ablation*). If removing a strategy from the full combination lowers the score clearly, the strategy really contributes.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

TOP3 = ["TotalArea", "HouseAge", "TotalBathrooms"]
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)


def build_features(ordinal=False, top3=False):
    """Features for one configuration (same rows, so get_split gives the same split)."""
    df = add_engineered_features(df_clean, TOP3 if top3 else [])
    if ordinal:
        df = ordinal_encode(df)
    return encode_features(df)


def evaluate_configuration(ordinal=False, top3=False, impute=False, outliers=False, weights=False):
    X_variant_train, _, y_variant_train, _ = get_split(build_features(ordinal, top3), y_all)
    return evaluate_strategy(X_variant_train, y_variant_train, cv=rcv, remove_outliers=outliers,
                             impute=impute, weights=weights)


full = dict(ordinal=True, top3=True, impute=True, outliers=True, weights=True)
configurations = {
    "Baseline": {},
    "+ Top 3 features": dict(top3=True),
    "+ Outlier removal": dict(outliers=True),
    "+ Median imputation": dict(impute=True),
    "+ Ordinal encoding": dict(ordinal=True),
    "+ Balanced weights": dict(weights=True),
    "All five": full,
    **{f"All five without {name}": {**full, name: False} for name in full},
    "Ordinal + Top 3 + weights": dict(ordinal=True, top3=True, weights=True),
}

rows = []
for name, settings in configurations.items():  # 13 configurations x 15 fits, takes a few minutes
    scores = evaluate_configuration(**settings)
    rows.append({"Configuration": name, "Mean macro-F1": scores.mean(), "Std": scores.std()})
    print(f"{name}: {scores.mean():.4f} +/- {scores.std():.4f}")

ablation = pd.DataFrame(rows).sort_values("Mean macro-F1", ascending=False).reset_index(drop=True)
display(ablation.round(4))

In [ ]:
plot_data = ablation.sort_values("Mean macro-F1")
baseline_mean = ablation.loc[ablation["Configuration"] == "Baseline", "Mean macro-F1"].iloc[0]

plt.figure(figsize=(9, 6))
plt.barh(plot_data["Configuration"], plot_data["Mean macro-F1"] - baseline_mean, color="tab:blue")
plt.axvline(0, color="black", linewidth=1)
plt.axvline(0.01, color="grey", linestyle="--", label="~0.01 = noise level (Part A)")
plt.xlabel("Difference in mean macro-F1 vs. baseline (repeated 5-fold CV)")
plt.title("Which data preparation strategies help?")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

**Interpretation** (the baseline is 0.7255 here instead of 0.7225 because the score is averaged over 15 instead of 5 folds):
- **Two strategies really matter: the top-3 features and the balanced weights.** Removing either of them from the full combination lowers the score by about 0.013 (0.7449 → 0.7319 without the features, 0.7333 without the weights).
- The weights alone help little (+0.003), but in combination with the other strategies they are essential: their effect depends on the features.
- **Outlier removal** (+0.000 alone; the combination is even slightly better without it) and **median imputation** (−0.005 alone) do not help and are dropped.
- **Ordinal encoding** adds a little (+0.006 alone; −0.003 when removed from the combination). This is close to the noise, but it is the more appropriate encoding and gives the most stable result together with the features and the weights.
- **Best configuration: Ordinal + Top 3 + weights = 0.7473 ± 0.015**, +0.022 compared with the baseline, clearly above the noise level and with the lowest spread of all configurations.

## 15. Do the Thresholds Generalize?

In Section 8, the thresholds were searched and evaluated on the same out-of-fold predictions, which is optimistic. Here, the thresholds are searched **only on the training part of each fold** (with an inner 5-fold CV) and then applied to the validation part, which the search has never seen. This is done for the baseline and for the best data preparation from Section 14.

In [ ]:
def nested_threshold_check(ordinal=False, top3=False, weights=False):
    """Thresholds are searched only on the training part of each fold and applied to its validation part."""
    X_variant_train, _, y_variant_train, _ = get_split(build_features(ordinal, top3), y_all)
    X_variant_train = X_variant_train.reset_index(drop=True)
    y_variant_train = y_variant_train.reset_index(drop=True)

    def fit(X_part, y_part):
        sample_weight = compute_sample_weight("balanced", y_part) if weights else None
        return XGBClassifier(random_state=42).fit(X_part, y_part, sample_weight=sample_weight)

    plain, adjusted = [], []
    for fold_train, fold_val in rcv.split(X_variant_train, y_variant_train):
        X_a = X_variant_train.iloc[fold_train].reset_index(drop=True)
        y_a = y_variant_train.iloc[fold_train].reset_index(drop=True)
        X_b, y_b = X_variant_train.iloc[fold_val], y_variant_train.iloc[fold_val]

        # Out-of-fold probabilities inside the training part -> search the thresholds there
        inner_prob = np.zeros((len(X_a), len(CLASS_NAMES)))
        for inner_train, inner_val in StratifiedKFold(5, shuffle=True, random_state=0).split(X_a, y_a):
            inner_prob[inner_val] = fit(X_a.iloc[inner_train], y_a.iloc[inner_train]).predict_proba(X_a.iloc[inner_val])
        thresholds, _ = search_thresholds(inner_prob, y_a)

        # Apply them to the validation part, which the search has never seen
        prob = fit(X_a, y_a).predict_proba(X_b)
        plain.append(f1_score(y_b, prob.argmax(axis=1), average="macro"))
        adjusted.append(f1_score(y_b, predict_with_thresholds(prob, thresholds), average="macro"))

    plain, adjusted = np.array(plain), np.array(adjusted)
    print(f"argmax: {plain.mean():.4f} | with thresholds: {adjusted.mean():.4f} | "
          f"difference: {np.mean(adjusted - plain):+.4f} | folds improved: {np.sum(adjusted > plain)}/{len(plain)}")


print("Baseline:")
nested_threshold_check()
print("\nOrdinal + Top 3 + weights:")
nested_threshold_check(ordinal=True, top3=True, weights=True)

**Interpretation:**
- For the **baseline**, thresholds bring a small real improvement on unseen data (+0.006, better in 10 of 15 folds), much less than the +0.021 suggested in Section 8.
- For the **final configuration with balanced weights**, thresholds make the result **worse** (−0.008, better in only 4 of 15 folds). The weights already shift the predictions towards the small classes; thresholds on top of them over-correct and mainly adapt to noise.
- **Conclusion:** the thresholds that maximize macro-F1 on the training data do not generalize when class weights are used. The final model therefore uses balanced weights and the normal argmax decision, without thresholds.

## 16. Final Data Preparation

Based on Section 14, the final data preparation consists of:

- the cleaned data from `lab2_utils.py`,
- the three engineered features `TotalArea`, `HouseAge` and `TotalBathrooms`,
- ordinal encoding for ordered categories, one-hot encoding for the remaining ones,
- balanced sample weights during training.

To see whether the improvement in macro-F1 also means better predictions for the minority classes, I compare the F1 score of every class for the baseline and the final preparation, using out-of-fold predictions on the training set (the test set stays untouched).

In [ ]:
def prepare_final_features(df):
    """Final data preparation of Part B: cleaned data + top 3 engineered features + ordinal encoding."""
    return encode_features(ordinal_encode(add_engineered_features(df, TOP3)))


X_final = prepare_final_features(df_clean)
X_final_train, X_final_test, y_final_train, y_final_test = get_split(X_final, y_all)
print("Final feature matrix:", X_final.shape)


def oof_predictions(X_tr, y_tr, weights=False):
    """Out-of-fold class predictions on the training set (same 5 folds as before)."""
    X_tr, y_tr = X_tr.reset_index(drop=True), y_tr.reset_index(drop=True)
    pred = np.zeros(len(y_tr), dtype=int)
    for fold_train, fold_val in cv5.split(X_tr, y_tr):
        sample_weight = compute_sample_weight("balanced", y_tr.iloc[fold_train]) if weights else None
        model = XGBClassifier(random_state=42).fit(X_tr.iloc[fold_train], y_tr.iloc[fold_train],
                                                   sample_weight=sample_weight)
        pred[fold_val] = model.predict(X_tr.iloc[fold_val])
    return y_tr, pred


y_true_base, pred_base = oof_predictions(X_train, y_train)
y_true_final, pred_final = oof_predictions(X_final_train, y_final_train, weights=True)

per_class = pd.DataFrame({
    "Baseline": f1_score(y_true_base, pred_base, average=None),
    "Final preparation": f1_score(y_true_final, pred_final, average=None),
}, index=CLASS_NAMES)
per_class.loc["Macro-F1"] = per_class.mean()
display(per_class.round(3))

per_class.drop(index="Macro-F1").plot(kind="bar", figsize=(9, 5), rot=0)
plt.ylabel("F1 (out-of-fold, training set)")
plt.title("F1 per price class: baseline vs. final data preparation")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

**Interpretation:**
- Macro-F1 (out-of-fold) rises from 0.723 to **0.746**.
- The largest improvement is in the smallest class **≥400k (0.717 → 0.784)**, followed by 250–400k (0.768 → 0.798). This is exactly what the class weights and `TotalArea` (which separates large, expensive houses better) should achieve.
- No class gets worse; the large class 130–180k stays the same (0.782).

## 17. Conclusions

**Evaluation of the baseline (test set):**
- Macro-F1 = 0.748, accuracy = 0.782. Almost all errors (122 of 128) happen between neighboring price classes.
- The weakest classes are ≥400k (F1 = 0.636, recall 0.538) and 100–130k (0.676). For ≥400k, the ranking is very good (AUC = 0.995), but the argmax decision misses many expensive houses, and the PR curve (AP = 0.776) shows how sensitive this small class is to false positives.
- Early stopping finds the best validation loss after only 22 boosting iterations, which confirms the overfitting of the default model (100 trees).

**Data preparation strategies (repeated stratified 5-fold CV, 15 fits each):**

| Strategy | Effect on macro-F1 | Decision |
|---|---|---|
| Top 3 engineered features (`TotalArea`, `HouseAge`, `TotalBathrooms`) | +0.012 alone, −0.013 when removed from the combination | **used** |
| Balanced sample weights | +0.003 alone, −0.012 when removed from the combination | **used** |
| Ordinal encoding of ordered categories | +0.006 alone, −0.003 when removed from the combination | **used** (appropriate encoding, most stable) |
| Outlier removal (> 4,000 sqft, training data only) | +0.000 | not used |
| Median imputation | −0.005 | not used (NaN handled by XGBoost) |
| Random oversampling | +0.000 (single 5-fold CV) | not used |
| Class-specific thresholds | −0.008 on unseen folds (with weights) | not used |

**Final data preparation:** ordinal encoding + top 3 engineered features + balanced sample weights; missing values are left as NaN and no outliers are removed.
- Mean macro-F1 (repeated CV): **0.747 ± 0.015**, compared with 0.726 for the baseline (+0.022).
- The largest gain is in the smallest class ≥400k (out-of-fold F1 0.717 → 0.784); no class gets worse.

**What we learned:**
- Differences of about 0.01 between single 5-fold CV results can be noise. Strategies have to be compared on the same folds, ideally with repeated CV.
- Strategies must also be tested in combination, because their effects do not simply add up.
- Every change to the data (removing outliers, imputing, balancing, searching thresholds) must be done only on the training part; otherwise the evaluation becomes optimistic or the test set changes.

**For Part C (tuning):** the hyperparameters should be optimized on this final data preparation, with balanced sample weights also inside the Optuna objective (`sample_weight=compute_sample_weight("balanced", y)` when fitting). The early-stopping result (22 iterations) suggests tuning `n_estimators` together with `learning_rate`.